In [ ]:
from pathlib import Path
import subprocess,sys,json,hashlib,base64,zipfile,io,importlib
out=Path("/kaggle/working/rename-acceptance");out.mkdir(exist_ok=True)
wheel=next(Path("/kaggle/input").rglob('multigpubeamsearch-0.1.0.dev1-py3-none-any.whl'))
assert hashlib.sha256(wheel.read_bytes()).hexdigest()=='d67910a1dc8bbee31cd11af75d39b2fa3062e1cae9bc5c1cf3d380f59ad9e7b5'
subprocess.run([sys.executable,"-m","pip","install","-q",str(wheel),"pytest"],check=True)
import cayleypy
original=cayleypy.CayleyGraph.beam_search
import multigpubeamsearch
import cayleypy_native
assert cayleypy.CayleyGraph.beam_search is original
assert multigpubeamsearch.NativeOptions is cayleypy_native.NativeOptions
for module in ["backend","bootstrap","build","contracts","dispatch","errors","hamming","models","options","preparation","results","sources"]:
    assert importlib.import_module("multigpubeamsearch."+module) is importlib.import_module("cayleypy_native."+module)
from importlib.metadata import version
assert version("multigpubeamsearch")=="0.1.0.dev1"
testdir=Path("/tmp/rename-check/integrations/cayleypy_native/tests");testdir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:z.extractall(testdir)
import os
os.environ["CAYLEYPY_NATIVE_TEST_SOURCE_DIR"]=str(Path(multigpubeamsearch.__file__).parent / "_native_source")
r=subprocess.run([sys.executable,"-m","pytest",str(testdir),"-q"],capture_output=True,text=True)
(out/"pytest.log").write_text(r.stdout+r.stderr)
(out/"result.json").write_text(json.dumps({"new_import":True,"legacy_identity":True,"import_does_not_patch":True,"distribution_version":version("multigpubeamsearch"),"pytest_returncode":r.returncode},indent=2))
print(r.stdout[-2500:]);assert r.returncode==0
